# FragPipe Multi-Plex Validation (Runs 1–5)

Loads the first N completed plex results and produces:
- **Per-plex histograms**: unweighted ratio + best-charge-state-per-peptide ratio
- **Cross-run box plot**: enrichment distribution with each plex as one box
- **Intensity trend**: ratio vs precursor intensity quintile, one panel per plex

In [ ]:
import glob
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

# ── CONFIG ────────────────────────────────────────────────────────────────────
PLEX_LIST_PATH = "/scratch/leduc.an/AAS_Evo/MS_SEARCH/plex_list.txt"
RESULTS_BASE   = "/scratch/leduc.an/AAS_Evo/MS_SEARCH/results"
REPO_DIR       = "/home/leduc.an/AAS_Evo_project/AAS_Evo"
TMT_MAP        = f"{REPO_DIR}/metadata/PDC_meta/pdc_file_tmt_map.tsv"
GDC_META       = f"{REPO_DIR}/metadata/GDC_meta/gdc_meta_matched.tsv"
FASTA_DIR      = "/scratch/leduc.an/AAS_Evo/FASTA/per_sample"
MISSENSE       = "/scratch/leduc.an/AAS_Evo/VEP/all_missense_mutations.tsv"
REF_FASTA      = "/scratch/leduc.an/AAS_Evo/SEQ_FILES/uniprot_human_canonical.fasta"
OUT_DIR        = "/scratch/leduc.an/AAS_Evo/MS_SEARCH/validation_multi"

N_PLEXES = 5   # first N plexes from plex_list.txt

os.makedirs(OUT_DIR, exist_ok=True)

with open(PLEX_LIST_PATH) as f:
    plex_ids = [l.strip() for l in f if l.strip()][:N_PLEXES]

print(f"Plexes to analyze ({len(plex_ids)}):")
for i, p in enumerate(plex_ids, 1):
    print(f"  {i}. {p}")

CHANNEL_ORDER = ["126","127N","127C","128N","128C","129N","129C","130N","130C","131N","131C"]
TMT_CHANNEL_MAP = {
    "tmt_126":"126",  "tmt_127n":"127N", "tmt_127c":"127C",
    "tmt_128n":"128N","tmt_128c":"128C", "tmt_129n":"129N",
    "tmt_129c":"129C","tmt_130n":"130N", "tmt_130c":"130C",
    "tmt_131":"131N", "tmt_131c":"131C",
}

In [ ]:
# ── LOAD SHARED RESOURCES (once, reused across all plexes) ───────────────────
AA3TO1 = {
    "Ala":"A","Arg":"R","Asn":"N","Asp":"D","Cys":"C","Gln":"Q","Glu":"E",
    "Gly":"G","His":"H","Ile":"I","Leu":"L","Lys":"K","Met":"M","Phe":"F",
    "Pro":"P","Ser":"S","Thr":"T","Trp":"W","Tyr":"Y","Val":"V",
}

def parse_hgvsp_to_swap(hgvsp):
    m = re.search(r'p\.([A-Z][a-z]{2})(\d+)([A-Z][a-z]{2})', str(hgvsp))
    if m:
        ref = AA3TO1.get(m.group(1))
        alt = AA3TO1.get(m.group(3))
        if ref and alt:
            return f"{ref}{m.group(2)}{alt}"
    return None

# gene symbol → UniProt accession (from GN= field in reference FASTA)
gene_to_acc = {}
with open(REF_FASTA) as f:
    for line in f:
        if line.startswith(">"):
            m = re.search(r'GN=(\S+)', line)
            if m:
                gene_to_acc[m.group(1)] = line.split("|")[1]
print(f"Gene → accession entries: {len(gene_to_acc):,}")

tmt = pd.read_csv(TMT_MAP, sep="\t")
gdc = pd.read_csv(GDC_META, sep="\t")
if "file_id" in gdc.columns and "gdc_file_id" not in gdc.columns:
    gdc = gdc.rename(columns={"file_id": "gdc_file_id"})
print(f"TMT map rows: {len(tmt):,}   GDC meta rows: {len(gdc):,}")

print("Loading missense table (shared filter base)...")
missense = pd.read_csv(MISSENSE, sep="\t", low_memory=False)
print(f"Total missense rows: {len(missense):,}")

In [ ]:
# ── GLOBAL MUTATION STATS (from full missense table, all samples) ─────────────
# For each unique mutation (accession, swap), compute:
#   n_samples_global  — how many distinct patients have this mutation called
#   avg_vaf_global    — mean VAF across those patients
#   median_vaf_global — median VAF
# This is the "population prevalence" of each mutation in our cohort.
# High avg_vaf (~0.5) + many samples → common germline variant (may have slipped gnomAD filter)
# Low avg_vaf + few samples          → rare patient-specific variant

missense_parsed = missense.copy()
missense_parsed["swap"]      = missense_parsed["HGVSp"].apply(parse_hgvsp_to_swap)
missense_parsed["accession"] = missense_parsed["SYMBOL"].map(gene_to_acc)
missense_parsed = missense_parsed.dropna(subset=["swap", "accession", "VAF"])
missense_parsed["VAF"] = pd.to_numeric(missense_parsed["VAF"], errors="coerce")
missense_parsed = missense_parsed.dropna(subset=["VAF"])

mut_global_stats = (
    missense_parsed
    .groupby(["accession", "swap"])
    .agg(
        n_samples_global  = ("sample_id", "nunique"),
        avg_vaf_global    = ("VAF", "mean"),
        median_vaf_global = ("VAF", "median"),
        std_vaf_global    = ("VAF", "std"),
    )
    .reset_index()
)

# Fast lookup dict: (accession, swap) → row as dict
mut_stats_lookup = {
    (r["accession"], r["swap"]): r
    for _, r in mut_global_stats.iterrows()
}

print(f"Global mutation stats computed: {len(mut_global_stats):,} unique mutations")
print(f"Samples in missense table:      {missense['sample_id'].nunique():,}")
print(f"\nVAF distribution across all mutations:")
print(mut_global_stats[["avg_vaf_global","median_vaf_global","n_samples_global"]].describe().round(3))

In [ ]:
# ── PROCESSING HELPERS ────────────────────────────────────────────────────────
PRECURSOR_CANDIDATES = ["Intensity", "Precursor Intensity", "MS1 Intensity",
                        "PrecursorIntensity", "precursor_intensity"]

def find_ri_cols(df):
    found = {}
    for ch in CHANNEL_ORDER:
        if ch in df.columns:
            found[ch] = ch; continue
        cands = [c for c in df.columns if c.startswith("Intensity") and c.endswith(f"_{ch}")]
        if cands:
            found[ch] = cands[0]; continue
        cands = [c for c in df.columns if c.startswith(ch) and "intensity" in c.lower()]
        if cands:
            found[ch] = cands[0]
    return found

def parse_mapped_proteins(mapped_str):
    if pd.isna(mapped_str):
        return set()
    pairs = set()
    for entry in str(mapped_str).split(","):
        parts = entry.strip().split("|")
        if len(parts) >= 2:
            pid_parts = parts[1].split("-")
            if len(pid_parts) >= 3:
                pairs.add((pid_parts[0], pid_parts[1]))
    return pairs

def parse_protein_id(protein_id):
    if pd.isna(protein_id):
        return set()
    parts = str(protein_id).split("-")
    if len(parts) >= 3:
        return {(parts[0], parts[1])}
    return set()


def process_plex(plex_id, same_patient_exclude=True):
    results_dir = os.path.join(RESULTS_BASE, plex_id)
    psm_matches = sorted(glob.glob(os.path.join(results_dir, "*_1/psm.tsv")))
    if not psm_matches:
        print(f"  [{plex_id[:40]}] No psm.tsv — skipping")
        return None

    psm = pd.read_csv(psm_matches[0], sep="\t", low_memory=False)
    ri_col_map = find_ri_cols(psm)
    ri_cols    = list(ri_col_map.values())

    mut_all = psm[psm["Entry Name"].str.endswith("-mut", na=False)].copy()
    mut_psm = mut_all[(mut_all[ri_cols].fillna(0) > 0).any(axis=1)].copy() \
              if ri_cols else mut_all.copy()

    precursor_col = next((c for c in PRECURSOR_CANDIDATES if c in mut_psm.columns), None)

    plex_tmt = (tmt[tmt["run_metadata_id"] == plex_id]
                [["tmt_channel","case_submitter_id","sample_type"]].drop_duplicates())
    plex_tmt = plex_tmt[~plex_tmt["case_submitter_id"].str.lower()
                        .isin(["ref","reference","pooled","pool","nan"])]
    plex_tmt["channel"] = plex_tmt["tmt_channel"].map(TMT_CHANNEL_MAP)

    plex_meta = plex_tmt.merge(
        gdc[["gdc_file_id","case_submitter_id","sample_type"]],
        on=["case_submitter_id","sample_type"], how="left")

    channel_to_case = {}
    for _, row in plex_meta.iterrows():
        if pd.notna(row.get("channel")) and pd.notna(row.get("case_submitter_id")):
            channel_to_case[row["channel"]] = row["case_submitter_id"]

    mutation_to_channels = defaultdict(set)
    channels_with_fastas = set()
    for _, row in plex_meta.iterrows():
        uuid, channel = row["gdc_file_id"], row["channel"]
        if pd.isna(uuid) or pd.isna(channel):
            continue
        fasta_path = os.path.join(FASTA_DIR, f"{uuid}_mutant.fasta")
        if not os.path.exists(fasta_path):
            continue
        channels_with_fastas.add(channel)
        with open(fasta_path) as f:
            for line in f:
                if not line.startswith(">"): continue
                parts = line[1:].strip().split("|")
                if len(parts) >= 4 and parts[0] == "mut":
                    mutation_to_channels[(parts[1], parts[3])].add(channel)

    uuid_to_channel = (plex_meta.dropna(subset=["gdc_file_id","channel"])
                                .set_index("gdc_file_id")["channel"].to_dict())
    plex_missense = missense[missense["sample_id"].isin(uuid_to_channel)]

    mutation_channel_vaf = {}
    for _, vrow in plex_missense.iterrows():
        vaf     = vrow.get("VAF", np.nan)
        channel = uuid_to_channel.get(vrow["sample_id"])
        if pd.isna(vaf) or not channel: continue
        acc  = gene_to_acc.get(str(vrow.get("SYMBOL", "")))
        swap = parse_hgvsp_to_swap(str(vrow.get("HGVSp", "")))
        if acc and swap:
            mutation_channel_vaf[(acc, swap, channel)] = float(vaf)

    rows = []
    n_same_patient_excluded = 0
    for _, row in mut_psm.iterrows():
        mutations = parse_mapped_proteins(row.get("Mapped Proteins", float("nan")))
        if not mutations:
            mutations = parse_protein_id(row.get("Protein ID", float("nan")))
        if not mutations: continue

        have_channels = set()
        for mk in mutations:
            have_channels |= mutation_to_channels.get(mk, set())
        have_channels &= channels_with_fastas

        not_have = channels_with_fastas - have_channels

        if same_patient_exclude:
            have_cases = {channel_to_case.get(ch) for ch in have_channels} - {None}
            same_patient_ch = {ch for ch in not_have
                               if channel_to_case.get(ch) in have_cases}
            not_have -= same_patient_ch
            n_same_patient_excluded += len(same_patient_ch)

        if not have_channels or not not_have: continue

        have_ri = [(ch, row[ri_col_map[ch]]) for ch in have_channels
                   if ch in ri_col_map and pd.notna(row[ri_col_map[ch]])]
        not_ri  = [row[ri_col_map[ch]] for ch in not_have
                   if ch in ri_col_map and pd.notna(row[ri_col_map[ch]])]
        if not have_ri or not not_ri: continue

        mean_not  = np.mean(not_ri)
        mean_have = np.mean([r for _, r in have_ri])
        ratio     = mean_have / mean_not if mean_not > 0 else np.nan

        weighted_pairs = []
        for ch, ri_val in have_ri:
            for acc, sw in mutations:
                v = mutation_channel_vaf.get((acc, sw, ch))
                if v is not None:
                    weighted_pairs.append((ri_val, v)); break
        if weighted_pairs:
            ris_v, wts_v = zip(*weighted_pairs)
            ratio_vaf = np.average(ris_v, weights=wts_v) / mean_not if mean_not > 0 else np.nan
        else:
            ratio_vaf = np.nan

        prec = float(row[precursor_col]) \
            if (precursor_col and pd.notna(row.get(precursor_col))
                and row.get(precursor_col, 0) > 0) else np.nan

        # ── Global VAF stats lookup (use first sorted mutation as key) ─────────
        first_mut = min(mutations) if mutations else (None, None)
        gstats    = mut_stats_lookup.get(first_mut, {})

        rows.append({
            "Peptide":             row.get("Peptide", ""),
            "accession":           first_mut[0] if first_mut[0] else "",
            "swap":                first_mut[1] if first_mut[1] else "",
            "ratio":               ratio,
            "ratio_vaf":           ratio_vaf,
            "precursor_intensity": prec,
            "n_have_ch":           len(have_channels),
            "n_not_have_ch":       len(not_have),
            "n_fasta_channels":    len(channels_with_fastas),
            "avg_vaf_global":      gstats.get("avg_vaf_global",    np.nan),
            "n_samples_global":    gstats.get("n_samples_global",  np.nan),
            "median_vaf_global":   gstats.get("median_vaf_global", np.nan),
        })

    ratios = pd.DataFrame(rows).dropna(subset=["ratio"])
    n_uniq = ratios.drop_duplicates(subset=["Peptide"])["ratio"].notna().sum()
    excl_str = f"  same-patient excluded: {n_same_patient_excluded:,}" if same_patient_exclude else ""
    print(f"  [{plex_id[:40]}]  "
          f"mut PSMs: {len(mut_psm):,}  →  ratios: {len(ratios):,}  "
          f"(uniq pep: {n_uniq:,},  FASTA ch: {len(channels_with_fastas)}{excl_str})")
    return ratios


# ── RUN BOTH VERSIONS ─────────────────────────────────────────────────────────
print("Processing plexes (without same-patient exclusion)...\n")
all_ratios = {}
for plex_id in plex_ids:
    df = process_plex(plex_id, same_patient_exclude=False)
    if df is not None and len(df) > 0:
        all_ratios[plex_id] = df

print(f"\nProcessing plexes (with same-patient exclusion)...\n")
all_ratios_excl = {}
for plex_id in plex_ids:
    df = process_plex(plex_id, same_patient_exclude=True)
    if df is not None and len(df) > 0:
        all_ratios_excl[plex_id] = df

print(f"\nDone: {len(all_ratios)} plexes (no exclusion) | "
      f"{len(all_ratios_excl)} plexes (with exclusion)")

In [ ]:
# ── DIAGNOSTIC: Where do matched tumor/normal channels actually land? ─────────
# For each plex, for patients with both tumor and normal channels:
#   - Are both channels in channels_with_fastas?
#   - Is the normal channel in have_channels or not_have for a given mutation?

print("Diagnosing same-patient channel assignment per plex...\n")

for plex_id in plex_ids:
    print(f"{'─'*70}")
    print(f"Plex: {plex_id}")

    plex_tmt = (tmt[tmt["run_metadata_id"] == plex_id]
                [["tmt_channel","case_submitter_id","sample_type"]].drop_duplicates())
    plex_tmt = plex_tmt[~plex_tmt["case_submitter_id"].str.lower()
                        .isin(["ref","reference","pooled","pool","nan"])]
    plex_tmt["channel"] = plex_tmt["tmt_channel"].map(TMT_CHANNEL_MAP)

    plex_meta = plex_tmt.merge(
        gdc[["gdc_file_id","case_submitter_id","sample_type"]],
        on=["case_submitter_id","sample_type"], how="left")

    channel_to_case  = {}
    channel_to_stype = {}
    channel_to_uuid  = {}
    for _, row in plex_meta.iterrows():
        ch = row.get("channel")
        if pd.isna(ch): continue
        case = row.get("case_submitter_id")
        uuid = row.get("gdc_file_id")
        st   = row.get("sample_type", "?")
        if pd.notna(case): channel_to_case[ch]  = case
        if pd.notna(uuid): channel_to_uuid[ch]  = uuid
        channel_to_stype[ch] = st

    # Which channels have FASTAs?
    channels_with_fastas = set()
    for ch, uuid in channel_to_uuid.items():
        if os.path.exists(os.path.join(FASTA_DIR, f"{uuid}_mutant.fasta")):
            channels_with_fastas.add(ch)

    # Build mutation_to_channels
    from collections import defaultdict
    mutation_to_channels = defaultdict(set)
    for ch in channels_with_fastas:
        uuid = channel_to_uuid[ch]
        with open(os.path.join(FASTA_DIR, f"{uuid}_mutant.fasta")) as f:
            for line in f:
                if not line.startswith(">"): continue
                parts = line[1:].strip().split("|")
                if len(parts) >= 4 and parts[0] == "mut":
                    mutation_to_channels[(parts[1], parts[3])].add(ch)

    # Find patients with >1 channel in plex (any sample type)
    from collections import defaultdict as dd2
    case_to_channels = dd2(list)
    for ch, case in channel_to_case.items():
        case_to_channels[case].append(ch)

    multi_channel_cases = {c: chs for c, chs in case_to_channels.items() if len(chs) > 1}

    print(f"  Patients with >1 channel in plex: {len(multi_channel_cases)}")
    for case, chs in multi_channel_cases.items():
        info = [(ch, channel_to_stype.get(ch,'?'),
                 'HAS_FASTA' if ch in channels_with_fastas else 'no_fasta')
                for ch in sorted(chs)]
        print(f"    {case}: {info}")

        # For this patient, check if both channels share mutations (both in have)
        # or if one is have and the other ends up in not_have
        for ch, st, fasta_status in info:
            if ch not in channels_with_fastas: continue
            uuid = channel_to_uuid.get(ch)
            # Count how many of this channel's mutations also appear in the other channel
            other_chs = [c for c, _, _ in info if c != ch and c in channels_with_fastas]
            if not other_chs: continue
            # Find mutations unique to this channel (not in other channels of same patient)
            unique_to_this = []
            shared = []
            for mk, mk_chs in mutation_to_channels.items():
                if ch in mk_chs:
                    others_have = any(oc in mk_chs for oc in other_chs)
                    if others_have:
                        shared.append(mk)
                    else:
                        unique_to_this.append(mk)
            print(f"      {ch} ({st}): {len(shared)} shared mutations, "
                  f"{len(unique_to_this)} unique → "
                  f"{'unique ones would put other channel in NOT-HAVE' if unique_to_this else 'all shared → other channel already in HAVE'}")

print(f"\n{'─'*70}")
print("KEY: 'unique → other channel in NOT-HAVE' means exclusion should fire.")
print("     'all shared → other channel in HAVE' means exclusion is redundant.")

In [ ]:
# ── PLOT 1: Per-plex histograms — 2 rows × N_PLEXES columns ──────────────────
# Row 0: all PSMs unweighted
# Row 1: best charge state per peptide (deduplicated by highest precursor intensity)

n = len(all_ratios)
plex_list_ordered = list(all_ratios.keys())

# Short label: pull tissue/study segment from plex ID
def short_label(plex_id):
    parts = plex_id.split("_")
    # e.g. "01CPTAC_CCRCC_Proteome_JHU_20171007" → "CCRCC JHU"
    if len(parts) >= 4:
        return f"{parts[1]}\n{parts[3]}"
    return plex_id[:20]

fig, axes = plt.subplots(2, n, figsize=(4.5 * n, 9), sharey=False)
if n == 1:
    axes = axes.reshape(2, 1)

for col, plex_id in enumerate(plex_list_ordered):
    ratios = all_ratios[plex_id]

    r_unw = ratios["ratio"].replace(0, np.nan).dropna()
    r_best = (ratios
        .sort_values("precursor_intensity", ascending=False, na_position="last")
        .drop_duplicates(subset=["Peptide"], keep="first")
        ["ratio"].replace(0, np.nan).dropna())

    for row_idx, (data, color, row_title) in enumerate([
        (r_unw,  "#4878d0", "All PSMs — Unweighted"),
        (r_best, "#9467bd", "Best charge state per peptide"),
    ]):
        ax = axes[row_idx, col]
        if len(data) == 0:
            ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
        else:
            b = np.logspace(np.log10(max(data.min(), 1e-3)), np.log10(data.max()), 50)
            ax.hist(data, bins=b, color=color, edgecolor="white", linewidth=0.4)
            med = data.median()
            ax.axvline(x=1,   color="grey",    linestyle="--", linewidth=1.2)
            ax.axvline(x=med, color="#e74c3c", linestyle="-",  linewidth=1.5,
                       label=f"med = {med:.2f}")
            ax.set_xscale("log")
            ax.legend(fontsize=8)

        ax.set_xlabel("have RI / not-have RI  [log scale]")
        ax.set_ylabel("N PSMs")
        header = f"{short_label(plex_id)}\n" if row_idx == 0 else ""
        ax.set_title(f"{header}{row_title}\n(n = {len(data):,})", fontsize=9)

fig.suptitle("Per-plex channel enrichment ratio", fontsize=12, y=1.01)
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "plex_histograms.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
# ── PLOT: Same-patient exclusion comparison — 2 rows × N_PLEXES columns ──────
# Row 0: without same-patient exclusion
# Row 1: with same-patient exclusion
# Lets you see directly whether excluding a patient's own other channels
# shifts the ratio distribution.

n = len(plex_list_ordered)

fig, axes = plt.subplots(2, n, figsize=(4.5 * n, 9), sharey=False)
if n == 1:
    axes = axes.reshape(2, 1)

for col, plex_id in enumerate(plex_list_ordered):
    for row_idx, (ratios_dict, color, row_title) in enumerate([
        (all_ratios,      "#4878d0", "Without same-patient exclusion"),
        (all_ratios_excl, "#e8703a", "With same-patient exclusion"),
    ]):
        ax = axes[row_idx, col]
        ratios = ratios_dict.get(plex_id)

        if ratios is None or len(ratios) == 0:
            ax.text(0.5, 0.5, "No data", ha="center", va="center",
                    transform=ax.transAxes)
        else:
            r = ratios["ratio"].replace(0, np.nan).dropna()
            b = np.logspace(np.log10(max(r.min(), 1e-3)), np.log10(r.max()), 50)
            ax.hist(r, bins=b, color=color, edgecolor="white", linewidth=0.4, alpha=0.85)
            med = r.median()
            ax.axvline(x=1,   color="grey",    linestyle="--", linewidth=1.2)
            ax.axvline(x=med, color="#e74c3c", linestyle="-",  linewidth=1.5,
                       label=f"med = {med:.2f}")
            ax.set_xscale("log")
            ax.legend(fontsize=8)

        ax.set_xlabel("have RI / not-have RI  [log scale]")
        ax.set_ylabel("N PSMs")
        header = f"{short_label(plex_id)}\n" if row_idx == 0 else ""
        n_psms = len(ratios_dict.get(plex_id, pd.DataFrame()))
        ax.set_title(f"{header}{row_title}\n(n = {n_psms:,})", fontsize=9)

fig.suptitle("Effect of same-patient exclusion on enrichment ratio", fontsize=12, y=1.01)
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "same_patient_exclusion_comparison.pdf"), bbox_inches="tight")
plt.show()

# ── Summary: median ratio before vs after ─────────────────────────────────────
print(f"\n{'Plex':<45} {'Med (no excl)':>14}  {'Med (excl)':>11}  {'Δ median':>9}")
print("-" * 85)
for plex_id in plex_list_ordered:
    r_before = all_ratios.get(plex_id, pd.DataFrame()).get("ratio", pd.Series()).replace(0, np.nan).dropna()
    r_after  = all_ratios_excl.get(plex_id, pd.DataFrame()).get("ratio", pd.Series()).replace(0, np.nan).dropna()
    med_b = r_before.median() if len(r_before) else float("nan")
    med_a = r_after.median()  if len(r_after)  else float("nan")
    delta = med_a - med_b
    print(f"{plex_id[:44]:<45} {med_b:>14.3f}  {med_a:>11.3f}  {delta:>+9.3f}")

In [ ]:
# ── PLOT 2: Cross-run summary box plot ────────────────────────────────────────
# Each plex = one box.  x-axis: plex (short label).  y-axis: ratio (log).
# Shows run-to-run consistency of enrichment.

fig, axes = plt.subplots(1, 2, figsize=(max(10, 2.5 * n), 5))

for ax, col, color, title_suffix in [
    (axes[0], "ratio",     "#4878d0", "All PSMs — Unweighted"),
    (axes[1], "ratio",     "#9467bd", "Best charge state per peptide"),
]:
    data_by_plex, tick_labels = [], []
    for plex_id in plex_list_ordered:
        ratios = all_ratios[plex_id]
        if col == "ratio":
            if title_suffix.startswith("Best"):
                r = (ratios
                     .sort_values("precursor_intensity", ascending=False, na_position="last")
                     .drop_duplicates(subset=["Peptide"], keep="first")
                     ["ratio"].replace(0, np.nan).dropna())
            else:
                r = ratios["ratio"].replace(0, np.nan).dropna()
        data_by_plex.append(r.values)
        tick_labels.append(
            f"{short_label(plex_id)}\nn={len(r):,}\nmed={r.median():.2f}")

    ax.boxplot(
        data_by_plex,
        patch_artist=True,
        showfliers=True,
        flierprops=dict(marker=".", markersize=2, alpha=0.15, color=color),
        medianprops=dict(color="#e74c3c", linewidth=2),
        boxprops=dict(facecolor=color, alpha=0.45),
        whiskerprops=dict(color="grey"),
        capprops=dict(color="grey"),
    )
    ax.axhline(y=1, color="grey",      linestyle="--", linewidth=1.2, label="ratio = 1")
    ax.axhline(y=2, color="lightgrey", linestyle=":",  linewidth=1.0, label="ratio = 2")
    ax.set_yscale("log")
    ax.set_xticks(range(1, n + 1))
    ax.set_xticklabels(tick_labels, fontsize=8)
    ax.set_ylabel("mean RI (have) / mean RI (not-have)  [log scale]")
    ax.set_title(title_suffix)
    ax.legend(fontsize=8)

fig.suptitle("Cross-run enrichment ratio — runs 1–5", fontsize=11, y=1.01)
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "cross_run_boxplot.pdf"), bbox_inches="tight")
plt.show()

# ── Summary statistics table ───────────────────────────────────────────────────
print(f"\n{'Plex':<45} {'N PSMs':>8}  {'N uniq pep':>11}  {'Median':>7}  "
      f"{'%>1':>6}  {'%>2':>6}  {'%>5':>6}  {'FASTA ch':>9}")
print("-" * 105)
for plex_id in plex_list_ordered:
    ratios = all_ratios[plex_id]
    r = ratios["ratio"].replace(0, np.nan).dropna()
    n_uniq = ratios.drop_duplicates(subset=["Peptide"])["ratio"].notna().sum()
    fch = int(ratios["n_fasta_channels"].iloc[0]) if "n_fasta_channels" in ratios else "?"
    print(f"{plex_id[:44]:<45} {len(r):>8,}  {n_uniq:>11,}  {r.median():>7.2f}  "
          f"{100*(r>1).mean():>5.1f}%  {100*(r>2).mean():>5.1f}%  "
          f"{100*(r>5).mean():>5.1f}%  {str(fch):>9}")

In [ ]:
# ── PLOT 3: Enrichment ratio vs precursor intensity — one panel per plex ──────
# Each panel: ratio binned into 5 intensity quintiles (box plots).
# Flat trend → background RI noise is not the main cause of diluted ratios.

fig, axes = plt.subplots(1, n, figsize=(4.5 * n, 5), sharey=True)
if n == 1:
    axes = [axes]

for ax, plex_id in zip(axes, plex_list_ordered):
    ratios  = all_ratios[plex_id]
    r_prec  = ratios.dropna(subset=["precursor_intensity"]).copy()
    r_prec  = r_prec[r_prec["precursor_intensity"] > 0]

    if len(r_prec) < 10:
        ax.text(0.5, 0.5, "Insufficient\ndata", ha="center", va="center",
                transform=ax.transAxes)
        ax.set_title(short_label(plex_id))
        continue

    log_intens = np.log10(r_prec["precursor_intensity"])
    try:
        r_prec = r_prec.copy()
        r_prec["intens_bin"] = pd.qcut(log_intens, q=5, duplicates="drop")
    except ValueError:
        r_prec["intens_bin"] = pd.qcut(log_intens, q=4, duplicates="drop")

    bin_order = list(r_prec["intens_bin"].cat.categories)
    data_by_bin, tick_labels = [], []
    for b in bin_order:
        grp = r_prec[r_prec["intens_bin"] == b]["ratio"].replace(0, np.nan).dropna()
        data_by_bin.append(grp.values if len(grp) else np.array([np.nan]))
        tick_labels.append(f"10^{b.left:.1f}–\n10^{b.right:.1f}\n(n={len(grp):,})")

    ax.boxplot(
        data_by_bin,
        patch_artist=True,
        showfliers=True,
        flierprops=dict(marker=".", markersize=2, alpha=0.2, color="#4878d0"),
        medianprops=dict(color="#e74c3c", linewidth=2),
        boxprops=dict(facecolor="#4878d0", alpha=0.45),
        whiskerprops=dict(color="grey"),
        capprops=dict(color="grey"),
    )
    ax.axhline(y=1, color="grey", linestyle="--", linewidth=1.2)
    ax.set_yscale("log")
    ax.set_xticks(range(1, len(bin_order) + 1))
    ax.set_xticklabels(tick_labels, fontsize=7.5)
    ax.set_xlabel("Precursor intensity — low → high", fontsize=9)
    ax.set_title(f"{short_label(plex_id)}\n(n = {len(r_prec):,} PSMs)", fontsize=9)

axes[0].set_ylabel("mean RI (have) / mean RI (not-have)  [log scale]")

fig.suptitle(
    "Enrichment ratio vs precursor intensity — per plex\n"
    "(flat trend → background noise not driving diluted ratios)",
    fontsize=11, y=1.03)
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "intensity_trend_per_plex.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
# ── PLOT 4: Enrichment ratio vs average VAF across all samples ────────────────
# Pool all plexes. For each PSM:
#   x = avg_vaf_global  (mean VAF of this mutation across ALL samples in the dataset)
#   y = ratio           (have-channel / not-have-channel enrichment)
#
# Expectation if germline:
#   - Mutations at VAF ~0.5 (consistent heterozygous germline) → may have lower ratios
#     if they appear in multiple plex channels (denominator inflation)
#   - Rare/low-VAF mutations → should be more patient-specific → higher ratios
#
# Also shown: ratio binned by n_samples_global (how many patients share the mutation)

pooled = pd.concat(all_ratios.values(), ignore_index=True)
pooled = pooled.dropna(subset=["ratio", "avg_vaf_global"])
pooled = pooled[pooled["ratio"] > 0]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ── Panel 1: Scatter ratio vs avg_vaf_global ──────────────────────────────────
ax = axes[0]
ax.scatter(pooled["avg_vaf_global"], pooled["ratio"],
           alpha=0.15, s=4, color="#4878d0", rasterized=True)
ax.set_yscale("log")
ax.axhline(1, color="grey", linestyle="--", linewidth=1)
ax.set_xlabel("Average VAF across all samples with this mutation", fontsize=10)
ax.set_ylabel("Enrichment ratio (have / not-have)  [log]", fontsize=10)
ax.set_title("Ratio vs global average VAF\n(all plexes pooled)", fontsize=10)

# Overlay median per VAF bin
vaf_bins = np.arange(0, 1.05, 0.05)
pooled["vaf_bin_mid"] = pd.cut(pooled["avg_vaf_global"], bins=vaf_bins).apply(
    lambda x: x.mid if not pd.isna(x) else np.nan)
bin_med = pooled.groupby("vaf_bin_mid")["ratio"].median().reset_index()
ax.plot(bin_med["vaf_bin_mid"], bin_med["ratio"],
        color="#e74c3c", linewidth=2, marker="o", markersize=4, label="Median per bin")
ax.legend(fontsize=8)

# ── Panel 2: Box plot — ratio by n_samples_global bins ───────────────────────
ax = axes[1]
# Bin n_samples_global: 1, 2-3, 4-10, >10
def sample_bin(n):
    if n == 1:   return "1 sample\n(unique)"
    if n <= 3:   return "2–3 samples"
    if n <= 10:  return "4–10 samples"
    return ">10 samples"

sample_bin_order = ["1 sample\n(unique)", "2–3 samples", "4–10 samples", ">10 samples"]
pooled["sample_bin"] = pooled["n_samples_global"].apply(sample_bin)

data_by_bin  = [pooled[pooled["sample_bin"] == b]["ratio"].replace(0, np.nan).dropna().values
                for b in sample_bin_order]
counts       = [len(d) for d in data_by_bin]

bp = ax.boxplot(
    data_by_bin,
    patch_artist=True,
    showfliers=True,
    flierprops=dict(marker=".", markersize=2, alpha=0.2, color="#4878d0"),
    medianprops=dict(color="#e74c3c", linewidth=2),
    boxprops=dict(facecolor="#4878d0", alpha=0.45),
    whiskerprops=dict(color="grey"),
    capprops=dict(color="grey"),
)
ax.set_yscale("log")
ax.axhline(1, color="grey", linestyle="--", linewidth=1)
ax.set_xticks(range(1, len(sample_bin_order) + 1))
ax.set_xticklabels([f"{b}\n(n={c:,})" for b, c in zip(sample_bin_order, counts)], fontsize=8)
ax.set_ylabel("Enrichment ratio  [log]", fontsize=10)
ax.set_title("Ratio vs number of patients\nwith the mutation (global)", fontsize=10)

# ── Panel 3: 2D histogram — avg_vaf vs ratio ─────────────────────────────────
ax = axes[2]
log_ratio = np.log10(pooled["ratio"].clip(lower=1e-3))
h = ax.hist2d(pooled["avg_vaf_global"], log_ratio,
              bins=[40, 40], cmap="Blues", cmin=1)
fig.colorbar(h[3], ax=ax, label="N PSMs")
ax.axhline(0, color="#e74c3c", linestyle="--", linewidth=1, label="ratio = 1")
ax.set_xlabel("Average VAF across all samples", fontsize=10)
ax.set_ylabel("log₁₀(ratio)", fontsize=10)
ax.set_title("Density: global avg VAF vs log ratio", fontsize=10)
ax.legend(fontsize=8)

fig.suptitle(
    "Enrichment ratio vs global average VAF\n"
    "Do mutations common across many patients (high avg VAF) show lower enrichment?",
    fontsize=11, y=1.03)
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "ratio_vs_global_vaf.pdf"), bbox_inches="tight")
plt.show()

# ── Summary table ─────────────────────────────────────────────────────────────
print("\nMedian ratio by sample frequency bin:")
print(f"{'Bin':<22} {'N PSMs':>8}  {'Median ratio':>13}  {'% ratio>1':>10}  {'% ratio>2':>10}")
print("-" * 68)
for b in sample_bin_order:
    grp = pooled[pooled["sample_bin"] == b]["ratio"].replace(0, np.nan).dropna()
    if len(grp) == 0: continue
    print(f"{b.replace(chr(10),' '):<22} {len(grp):>8,}  {grp.median():>13.3f}  "
          f"{100*(grp>1).mean():>9.1f}%  {100*(grp>2).mean():>9.1f}%")